# M3-T01: tumor volume per region (cm3)

Volume = number of voxels x voxel volume (read from each file's NIfTI header), reported in cm3. Regions follow `bmts.common.constants.REGIONS`: TC = labels 1+3, WT = labels 1+2+3, ET = label 3 (they overlap).

Input: nnU-Net predictions for the 188 validation patients (M2-T04). Output: `results/M3-T01/tumor_volumes_val.csv`.

Before the PR is merged, the code only exists on the author's fork: set `REPO` to the fork URL and `BRANCH` to the branch name (`M3-T01-tumor-volume`). After the merge keep `REPO` as is and use `BRANCH = 'main'`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO = 'https://github.com/nhahub/NHA-5-187.git'
BRANCH = 'main'

!git clone -q -b {BRANCH} {REPO} /content/repo
!pip -q install nibabel pytest
!cd /content/repo && python -m pytest tests/quantitative -q

In [ ]:
import sys, glob
sys.path.insert(0, '/content/repo/src')
from bmts.quantitative.volumes import volumes_table

PRED_DIR = '/content/drive/MyDrive/BrainMRI_Data/runs/M2-T04_nnunet/pred_nnunet_val'
files_ = sorted(glob.glob(PRED_DIR + '/*.nii.gz'))
df = volumes_table(files_)
print(len(df), 'cases')
display(df.head())
print(df.describe().round(2).T[['min', 'mean', 'max']])

In [ ]:
# Sanity checks: every voxel is 1 mm3 here, and the regions are nested (ET <= TC <= WT)
assert len(df) == 188
assert (df.voxel_mm3 == 1.0).all()
assert (df.WT_voxels >= df.TC_voxels).all() and (df.TC_voxels >= df.ET_voxels).all()
print('sanity checks passed')

In [ ]:
# Check 3 cases with an independent count (no helper functions, divide by 1000 by hand)
import nibabel as nib, numpy as np

for f in files_[:3]:
    arr = np.asarray(nib.load(f).dataobj)
    wt = int((arr > 0).sum())
    tc = int(((arr == 1) | (arr == 3)).sum())
    et = int((arr == 3).sum())
    case = f.split('/')[-1].replace('.nii.gz', '')
    row = df[df.case == case].iloc[0]
    print(case)
    print('  WT:', wt / 1000, 'vs', row.WT_cm3, np.isclose(wt / 1000, row.WT_cm3))
    print('  TC:', tc / 1000, 'vs', row.TC_cm3, np.isclose(tc / 1000, row.TC_cm3))
    print('  ET:', et / 1000, 'vs', row.ET_cm3, np.isclose(et / 1000, row.ET_cm3))

In [ ]:
df.to_csv('/content/tumor_volumes_val.csv', index=False)
from google.colab import files
files.download('/content/tumor_volumes_val.csv')